In [ ]:
import numpy as np
import sympy
import json
import scipy.linalg
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path

def compute_terwilliger_classes(A_adj, base_vertex=0, target_dim=None, tol=1e-9, out_filename="t_algebra_classes.json"):
    """
    A_adj: The Adjacency matrix of the graph (NumPy array).
    base_vertex: The 0-indexed base vertex x.
    target_dim: The expected dim T (e.g., 49 for 8-cage, 76 for Dodecahedron). Stops when reached.
    out_filename: The name of the JSON file to save to.
    """
    n = A_adj.shape[0]
    
    # 1. BFS Distance Matrix
    dist = np.full((n, n), -1)
    for i in range(n):
        dist[i, i] = 0
        queue = [i]
        while queue:
            curr = queue.pop(0)
            for neighbor in np.where(A_adj[curr] == 1)[0]:
                if dist[i, neighbor] == -1:
                    dist[i, neighbor] = dist[i, curr] + 1
                    queue.append(neighbor)
                    
    dist_x = dist[base_vertex]
    D = np.max(dist_x)
    
    # 2. Generators (Omitting Identity since I = sum(E_i^*))
    E_star = [(np.diag((dist_x == i).astype(float)), f"E_{i}^*") for i in range(D + 1)]
    generators = [(A_adj, "A")] + E_star
    
    # 3. Find Linearly Independent Words
    basis_gs = []
    basis_mats = []
    basis_words = []
    
    def try_add(M, word):
        v = M.flatten()
        if np.linalg.norm(v) < tol: return False
        r = v.copy()
        for b in basis_gs:
            r -= np.dot(r, b) * b
        norm = np.linalg.norm(r)
        if norm > tol:
            basis_gs.append(r / norm)
            basis_mats.append(M)
            basis_words.append(word)
            return True
        return False

    # Seed strictly with E_i^*
    for mat, name in E_star:
        try_add(mat, name)
        
    current_frontier = list(zip(basis_mats, basis_words))
    word_len = 1
    
    dim_str = f"Target Dimension {target_dim}" if target_dim else "Unknown target dimension"
    print(f"Finding basis words ({dim_str})...")
    
    while current_frontier and (target_dim is None or len(basis_words) < target_dim):
        new_frontier = []
        for M, M_word in current_frontier:
            for g_mat, g_name in generators:
                product_mat = np.array(M) @ g_mat
                product_word = f"{M_word}{g_name}"
                
                if try_add(product_mat, product_word):
                    new_frontier.append((product_mat, product_word))
                    if target_dim and len(basis_words) == target_dim: break
            if target_dim and len(basis_words) == target_dim: break
        current_frontier = new_frontier
        word_len += 1

    actual_dim = len(basis_words)
    print(f"Algebra closed/stopped at Dimension {actual_dim}. Row-reducing to find disjoint classes...")
    
    # 4. RREF Augmentation to Extract Canonical Classes & Linear Combinations
    # Note: np.eye size dynamically matches actual_dim (e.g., 76)
    stack = np.array([M.flatten() for M in basis_mats])
    aug_stack = np.hstack((stack, np.eye(actual_dim)))
    sym_aug = sympy.Matrix(aug_stack)
    
    rref_aug, _ = sym_aug.rref(iszerofunc=lambda x: abs(x) < tol)
    
    canonical_flat = np.array(rref_aug[:, :n*n], dtype=float)
    transform_matrix = np.array(rref_aug[:, n*n:], dtype=float)

    classes = []
    cell_labels = [["" for _ in range(n)] for _ in range(n)]

    # Handles unlimited labeling correctly (A..Z, AA..AZ, BA..BZ, etc.)
    def get_label(idx):
        L = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        if idx < 26: return L[idx]
        if idx < 26 + 26**2: return L[(idx//26)-1] + L[idx%26]
        return L[((idx - 26) // 26**2) - 1] + L[((idx - 26) % 26**2) // 26] + L[idx % 26]

    for i in range(actual_dim):
        lbl = get_label(i)
        
        coeffs = transform_matrix[i]
        lin_comb = []
        for c_idx, c_val in enumerate(coeffs):
            if abs(c_val) > tol:
                val_str = f"{c_val:.2f}".rstrip('0').rstrip('.') if c_val not in (1.0, -1.0) else ("" if c_val == 1.0 else "-")
                lin_comb.append(f"{val_str}{basis_words[c_idx]}")
        lin_comb_str = " + ".join(lin_comb).replace("+ -", "- ")

        mat_2d = canonical_flat[i].reshape((n, n))
        for r in range(n):
            for c in range(n):
                if abs(mat_2d[r, c]) > tol:
                    cell_labels[r][c] = lbl

        classes.append({
            "label": lbl,
            "linear_combination": lin_comb_str,
            "matrix": mat_2d.tolist()
        })

    output = {
        "n_vertices": n,
        "dimension": actual_dim,
        "cell_labels": cell_labels, 
        "classes": classes
    }

    with open(out_filename, "w") as f:
        json.dump(output, f)
        
    print(f"Successfully exported '{out_filename}'")


if __name__ == "__main__":
    # --- Graph Setup ---
    N = 126
    # 1. Setup Original Matrix A (126x126 Adjacency Matrix)
    edges = [
      [1,2], [1,3], [1,4], [2,5], [2,6], [3,7], [3,8], [4,9], [4,10],
      [5,11], [5,12], [6,13], [6,14], [7,15], [7,16], [8,17], [8,18],
      [9,19], [9,20], [10,21], [10,22], [11,23], [11,24], [12,25], [12,26],
      [13,27], [13,28], [14,29], [14,30], [15,31], [15,32], [16,33], [16,34],
      [17,35], [17,36], [18,37], [18,38], [19,39], [19,40], [20,41], [20,42],
      [21,43], [21,44], [22,45], [22,46], [23,47], [23,48], [24,49], [24,50],
      [25,51], [25,52], [26,53], [26,54], [27,55], [27,56], [28,57], [28,58],
      [29,59], [29,60], [30,61], [30,62], [31,63], [31,64], [32,65], [32,66],
      [33,67], [33,68], [34,69], [34,70], [35,71], [35,72], [36,73], [36,74],
      [37,75], [37,76], [38,77], [38,78], [39,79], [39,80], [40,81], [40,82],
      [41,83], [41,84], [42,85], [42,86], [43,87], [43,88], [44,89], [44,90],
      [45,91], [45,92], [46,93], [46,94], [47,95], [47,96], [48,97], [48,98],
      [49,99], [49,100], [50,101], [50,102], [51,103], [51,104], [52,105], [52,106],
      [53,107], [53,108], [54,109], [54,110], [55,111], [55,112], [56,113], [56,114],
      [57,115], [57,116], [58,117], [58,118], [59,119], [59,120], [60,121], [60,122],
      [61,123], [61,124], [62,125], [62,126], [63,107], [63,119], [64,98], [64,117],
      [65,100], [65,124], [66,104], [66,111], [67,96], [67,115], [68,110], [68,121],
      [69,106], [69,114], [70,102], [70,126], [71,109], [71,118], [72,95], [72,120],
      [73,103], [73,125], [74,99], [74,113], [75,101], [75,112], [76,105], [76,123],
      [77,97], [77,122], [78,108], [78,116], [79,101], [79,119], [80,103], [80,115],
      [81,109], [81,124], [82,97], [82,114], [83,108], [83,126], [84,95], [84,111],
      [85,105], [85,117], [86,99], [86,121], [87,106], [87,120], [88,100], [88,116],
      [89,110], [89,112], [90,98], [90,125], [91,107], [91,113], [92,96], [92,123],
      [93,102], [93,118], [94,104], [94,122]
    ]
    N = 126
    A_orig = np.zeros((N, N))
    for u, v in edges:
        A_orig[u-1, v-1] = 1
        A_orig[v-1, u-1] = 1
    
    # 2. Permute A for base vertex 1
    base_vertex_idx = 1
    graph = csr_matrix(A_orig)
    dist_matrix = shortest_path(csgraph=graph, directed=False, indices=base_vertex_idx)
    new_order = np.argsort(dist_matrix)
    A = A_orig[new_order, :][:, new_order]
    A = np.array(A, dtype=float).reshape(N, N)

    compute_terwilliger_classes(A, base_vertex=0, target_dim=168, out_filename="12cage_deux_classes.json")